# Démonstration de l'évaluation avant LoRA

Sur les 20 cas indépendants, Qwen reçoit l'énoncé et la consigne JSON.
Le corrigé sert à noter la réponse après génération.

Le code de génération et de notation est dans
[evaluate.py](src/lp_llm/evaluate.py). Ouvrir ce notebook à la racine du dépôt.
Par défaut, il lit l'expérience archivée dans `runs/base_independent_v1.json`.
Mettre `RELANCER = True` pour refaire la mesure en mémoire avec le modèle en
cache et un GPU CUDA.

In [1]:
import json
import sys
from hashlib import sha256
from pathlib import Path

root = Path.cwd()
sys.path.insert(0, str(root / "src"))

dataset = root / "data/human_test.jsonl"
archive = root / "runs/base_independent_v1.json"
RELANCER = False

if RELANCER:
    from lp_llm.evaluate import evaluate

    run = evaluate(dataset, root / ".hf_cache/hub")
else:
    run = json.loads(archive.read_text(encoding="utf-8"))

content = dataset.read_text(encoding="utf-8")
assert sha256(content.encode("utf-8")).hexdigest() == run["dataset_sha256_utf8_lf"]
cases = {case["id"]: case for case in map(json.loads, content.splitlines())}
predictions = {row["id"]: row for row in run["predictions"]}
assert cases.keys() == predictions.keys()

## Les quatre mesures

La conformité couvre le JSON et ses champs. L'exactitude compare la formulation
au corrigé. Le dernier contrôle teste la solution de la prédiction sur le
problème du corrigé.

In [2]:
labels = {
    "format": "JSON conforme au schéma",
    "formulation": "Formulation exacte",
    "solve": "Résolution du problème prédit",
    "reference": "Solution correcte sur le problème de référence",
}

print(run["model"], run["revision"])
for key, label in labels.items():
    print(f"{label} — {run['scores'][key]}/{run['total']}")

Qwen/Qwen2.5-1.5B-Instruct 989aa7980e4cf806f80c7fef2b1adb7bc71aa306
JSON conforme au schéma — 0/20
Formulation exacte — 0/20
Résolution du problème prédit — 0/20
Solution correcte sur le problème de référence — 0/20


## Un cas à examiner

Dans l'exécution archivée, l'énoncé sur les barbotines a donné une formulation
exacte à l'intérieur d'un bloc Markdown. Le contrôle strict rejette la réponse
entière. La dernière cellule évalue le contenu du bloc à titre de diagnostic.

In [3]:
from lp_llm.verify import verify_lp

case_id = "independent_production_006"
case = cases[case_id]
row = predictions[case_id]

print("Énoncé")
print(case["statement"])
print("\nRéponse brute")
print(row["raw_response"])
print("\nJSON de référence")
print(json.dumps(case["reference"], indent=2))

Énoncé
A ceramics plant schedules two kinds of slip. Let x1 be litres of white slip and x2 litres of grey slip, both nonnegative and continuous. Processing capacity is 112 minutes and water availability is 82 L. White slip uses 3 minutes and 1 L of water per litre made. Grey slip uses 2 minutes and 2 L of water per litre made. At most 38 L of the two slips combined can be stored. Each litre earns EUR 8 for white and EUR 7 for grey in profit. Formulate the profit-maximizing linear program.

Réponse brute
```json
{
  "variables": ["x1", "x2"],
  "objective": {
    "sense": "max",
    "coefficients": [8, 7]
  },
  "constraints": [
    {
      "coefficients": [3, 2],
      "sense": "<=",
      "rhs": 112
    },
    {
      "coefficients": [1, 2],
      "sense": "<=",
      "rhs": 82
    },
    {
      "coefficients": [1, 1],
      "sense": "<=",
      "rhs": 38
    }
  ],
  "bounds": [[0, null], [0, null]]
}
```

JSON de référence
{
  "variables": [
    "x1",
    "x2"
  ],
  "objective": {

In [4]:
print("Note enregistrée", {key: row[key] for key in labels})
print("Diagnostic strict", row["issues"])

lines = row["raw_response"].strip().splitlines()
if lines and lines[0] == "```json" and lines[-1] == "```":
    block = "\n".join(lines[1:-1])
    diagnostic = verify_lp(block, case["reference"])
    print("Contenu du bloc seul", {key: diagnostic[key] for key in labels})

Note enregistrée {'format': False, 'formulation': False, 'solve': False, 'reference': False}
Diagnostic strict ['JSON — Invalid JSON: Expecting value: line 1 column 1 (char 0)']
Contenu du bloc seul {'format': True, 'formulation': True, 'solve': True, 'reference': True}


## Cas sans le mot « linear »

In [5]:
for case_id, case in cases.items():
    if "linear program" not in case["statement"]:
        row = predictions[case_id]
        print(case_id, {key: row[key] for key in ("format", "formulation")})

independent_mixing_004 {'format': False, 'formulation': False}
independent_mixing_009 {'format': False, 'formulation': False}
independent_mixing_010 {'format': False, 'formulation': False}
